# Lecture: Generative Adversarial Networks (GANs)

In the previous notebook we saw that the VAE produces **blurry** samples. The reason is the MSE reconstruction loss: it penalises the average pixel-wise distance to the training image, which forces the decoder to produce the *mean* over all plausible outputs — a blurry compromise.

A **Generative Adversarial Network** (Goodfellow et al., 2014) takes a fundamentally different approach. Instead of a fixed loss function, it introduces a second network — the **Discriminator** $D$ — that acts as a learned critic:

- **Generator** $G$: maps a noise vector $z \sim \mathcal{N}(0, I)$ to a fake image $G(z)$.
- **Discriminator** $D$: maps an image $x$ to the probability $D(x) \in [0, 1]$ that it is real.

The two networks play a **minimax game**:

$$\min_G \max_D \; \mathbb{E}_{x \sim p_{data}}[\log D(x)] + \mathbb{E}_{z \sim p(z)}[\log(1 - D(G(z)))]$$

- $D$ tries to **maximise** the objective: output high values for real images, low values for fakes.
- $G$ tries to **minimise** the objective: produce images that $D$ cannot distinguish from real ones.

In practice the generator minimises $-\mathbb{E}[\log D(G(z))]$ (non-saturating loss) instead of $\mathbb{E}[\log(1 - D(G(z)))]$, which provides stronger gradients early in training when $D$ easily rejects all fakes.

The key insight: $D$ never sees a pixel-wise distance — it only judges *realism*. This pushes $G$ to produce **sharp, realistic-looking images** rather than blurry averages.

Run the following cell only if you are working with Google Colab to copy the required .py file into the root directory. If you are working locally, ignore this cell.

In [ ]:
!git clone https://github.com/Fjoelsak/AIBIP.git
!cp AIBIP/06-Generative_Image_Models/GAN.py ./
!cp AIBIP/06-Generative_Image_Models/VAE.py ./

### Data Preparation

We train on **Fashion-MNIST** — the same dataset used at the end of the VAE notebook. This allows a direct visual comparison of sample quality between VAE and GAN.

Images are normalised to $[-1, 1]$ to match the generator's Tanh output activation.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import FashionMNIST
from GAN import GAN

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

LATENT_DIM = 64
CHANNELS   = 64
BATCH_SIZE = 256
EPOCHS     = 30
LR         = 2e-4

FASHION_CLASSES = [
    "T-shirt", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal",  "Shirt",   "Sneaker",  "Bag",   "Ankle boot"
]

# Normalise to [-1, 1] to match the generator's Tanh output
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = FashionMNIST(root="./data", train=True, download=True, transform=transform)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)

print(f"Training samples: {len(train_dataset)}")

### Model Architecture

**Generator** $G$: Takes a noise vector $z \in \mathbb{R}^{64}$ and produces a 28×28 image via a linear projection followed by three transposed convolutional blocks. Tanh constrains output to $[-1, 1]$.

**Discriminator** $D$: Takes a 28×28 image and outputs a single probability via three strided convolutional blocks and a linear head. LeakyReLU (slope 0.2) is used instead of ReLU — a standard choice for discriminators that prevents dead neurons without the mode-collapse risk associated with ReLU.

Note that the discriminator has **no BatchNorm in the first layer**: adding normalisation before the first conv would destroy the scale information that helps $D$ detect generated images early in training.

In [ ]:
model = GAN(latent_dim=LATENT_DIM, channels=CHANNELS).to(device)

# Verify output shapes
_z    = torch.randn(4, LATENT_DIM).to(device)
_fake = model.generator(_z)
_prob = model.discriminator(_fake)

print("Generator output shape:     ", _fake.shape)  # (4, 1, 28, 28)
print("Discriminator output shape: ", _prob.shape)  # (4, 1)

n_params_g = sum(p.numel() for p in model.generator.parameters())
n_params_d = sum(p.numel() for p in model.discriminator.parameters())
print(f"Generator parameters:       {n_params_g:,}")
print(f"Discriminator parameters:   {n_params_d:,}")

### Training

GAN training alternates between two update steps per mini-batch:

**Step 1 — Update Discriminator:**
$$\mathcal{L}_D = -\mathbb{E}[\log D(x)] - \mathbb{E}[\log(1 - D(G(z)))]$$
Real images should be classified as real ($D(x) \to 1$), generated images as fake ($D(G(z)) \to 0$).

**Step 2 — Update Generator:**
$$\mathcal{L}_G = -\mathbb{E}[\log D(G(z))]$$
The generator is updated with the discriminator weights **frozen** (`detach()` on the fake images prevents gradients from flowing back into $D$). $G$ learns to produce images that $D$ classifies as real.

Both losses are logged separately each epoch. In a well-training GAN, $\mathcal{L}_D \approx \log 2 \approx 0.69$ at equilibrium — $D$ cannot do better than random guessing.

In [ ]:
criterion = nn.BCELoss()

opt_d = optim.Adam(model.discriminator.parameters(), lr=LR, betas=(0.5, 0.999))
opt_g = optim.Adam(model.generator.parameters(),     lr=LR, betas=(0.5, 0.999))

# Fixed noise for tracking generator progress across epochs
fixed_z = torch.randn(16, LATENT_DIM, device=device)

history_d, history_g = [], []

for epoch in range(EPOCHS):
    model.train()
    total_d = total_g = 0

    for real, _ in train_loader:
        real = real.to(device, non_blocking=True)
        B    = real.size(0)

        real_labels = torch.ones(B,  1, device=device)
        fake_labels = torch.zeros(B, 1, device=device)

        # ---- Step 1: Update Discriminator --------------------------------
        z    = torch.randn(B, LATENT_DIM, device=device)
        fake = model.generator(z).detach()  # detach: no gradient into G

        loss_d_real = criterion(model.discriminator(real), real_labels)
        loss_d_fake = criterion(model.discriminator(fake), fake_labels)
        loss_d      = (loss_d_real + loss_d_fake) / 2

        opt_d.zero_grad()
        loss_d.backward()
        opt_d.step()

        # ---- Step 2: Update Generator ------------------------------------
        z    = torch.randn(B, LATENT_DIM, device=device)
        fake = model.generator(z)

        # Generator wants D to output 1 (real) for its fakes
        loss_g = criterion(model.discriminator(fake), real_labels)

        opt_g.zero_grad()
        loss_g.backward()
        opt_g.step()

        total_d += loss_d.item()
        total_g += loss_g.item()

    n = len(train_loader)
    history_d.append(total_d / n)
    history_g.append(total_g / n)
    print(f"Epoch {epoch+1:3d}  loss_D={history_d[-1]:.4f}  loss_G={history_g[-1]:.4f}")

In [ ]:
model.save_model()

If you do not want to train, load the pre-trained model (latent_dim=64, 50 epochs, full Fashion-MNIST training set).

In [ ]:
model = GAN(latent_dim=LATENT_DIM, channels=CHANNELS).to(device)
model.load_model(path="AIBIP/06-Generative_Image_Models/models/gan_fashion_mnist.pth", device=device)

### Training Dynamics

Unlike VAE training, GAN losses do **not** simply decrease over time. The discriminator and generator are constantly adapting to each other, which produces characteristic oscillating loss curves.

At equilibrium, theory predicts $\mathcal{L}_D \approx \log 2 \approx 0.693$ — the point where the discriminator cannot distinguish real from fake better than random chance. In practice this equilibrium is rarely perfectly stable.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history_d, label="Discriminator loss $\\mathcal{L}_D$")
ax.plot(history_g, label="Generator loss $\\mathcal{L}_G$")
ax.axhline(np.log(2), color="gray", linestyle="--", label="Equilibrium ($\\log 2 \\approx 0.693$)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("GAN Training Dynamics — Fashion-MNIST")
ax.legend()
plt.tight_layout()
plt.show()

### Generated Samples

We sample 16 noise vectors from $p(z) = \mathcal{N}(0, I)$ and decode them with the trained generator. Images are rescaled from $[-1, 1]$ to $[0, 1]$ for display.

Compare these to the VAE samples from the previous notebook: GAN samples are **sharper** and show crisper edges and textures. The trade-off is that GANs offer no latent space with the structured properties of the VAE posterior.

In [ ]:
model.eval()

samples = model.generate(16, device=device).cpu()
samples = (samples + 1) / 2  # [-1, 1] -> [0, 1]

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i].squeeze(), cmap="gray")
    ax.axis("off")

plt.suptitle("GAN samples from $p(z) = \\mathcal{N}(0, I)$ — Fashion-MNIST", y=1.02)
plt.tight_layout()
plt.show()

### VAE vs. GAN — Direct Comparison

Side-by-side comparison of samples from the Fashion-MNIST VAE (notebook 65) and the GAN trained above.

| | VAE | GAN |
|---|---|---|
| Sample quality | Blurry (MSE averages over pixels) | Sharp (adversarial loss rewards realism) |
| Latent space | Structured, continuous, interpolatable | Unstructured — no posterior to inspect |
| Training | Stable, single objective | Unstable, two competing objectives |
| Mode coverage | Good (KL regularisation) | Risk of mode collapse |

Run this cell after loading both the VAE and GAN pre-trained models.

In [ ]:
from VAE import VAE

vae = VAE(latent_dim=2).to(device)
vae.load_model(path="AIBIP/06-Generative_Image_Models/models/vae_fashion_mnist.pth", device=device)
vae.eval()

n = 8
vae_samples = vae.sample(n, device=device).cpu()
gan_samples = ((model.generate(n, device=device) + 1) / 2).cpu()

fig, axes = plt.subplots(2, n, figsize=(14, 4))

for i in range(n):
    axes[0, i].imshow(vae_samples[i].squeeze(), cmap="gray")
    axes[0, i].axis("off")
    axes[1, i].imshow(gan_samples[i].squeeze(), cmap="gray")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("VAE", fontsize=12)
axes[1, 0].set_ylabel("GAN", fontsize=12)
plt.suptitle("VAE vs. GAN — Fashion-MNIST samples", fontsize=12)
plt.tight_layout()
plt.show()

### Latent Space Interpolation

Even without a structured posterior, we can interpolate linearly between two noise vectors $z_a$ and $z_b$ in the generator's input space. Because the generator is a continuous function, nearby latent points decode to visually similar images.

This is less principled than VAE interpolation — there is no guarantee that the path between $z_a$ and $z_b$ stays in a region of high data density — but in practice it often produces smooth and meaningful transitions.

In [ ]:
model.eval()

torch.manual_seed(0)
z_a = torch.randn(1, LATENT_DIM, device=device)
z_b = torch.randn(1, LATENT_DIM, device=device)

n_steps = 10
alphas  = torch.linspace(0, 1, n_steps)

fig, axes = plt.subplots(1, n_steps, figsize=(15, 2))

with torch.no_grad():
    for i, alpha in enumerate(alphas):
        z_interp = (1 - alpha) * z_a + alpha * z_b
        img = model.generator(z_interp).squeeze().cpu()
        img = (img + 1) / 2  # [-1, 1] -> [0, 1]
        axes[i].imshow(img, cmap="gray")
        axes[i].axis("off")
        axes[i].set_title(f"{alpha:.1f}", fontsize=8)

plt.suptitle("Latent space interpolation — GAN Generator", y=1.05)
plt.tight_layout()
plt.show()

### Mode Collapse

A fundamental failure mode of GANs is **mode collapse**: the generator learns to produce only a small subset of the training distribution — often a single convincing image — because this is sufficient to fool the discriminator.

Signs in the loss curves:
- $\mathcal{L}_G$ drops sharply while $\mathcal{L}_D$ spikes — the generator found a fixed point that fools $D$.
- Generated samples show little diversity across different noise inputs.

We can detect diversity collapse by measuring the **pairwise pixel variance** across a batch of samples: low variance indicates the generator is producing near-identical images.

In [ ]:
model.eval()

n_check  = 64
z_check  = torch.randn(n_check, LATENT_DIM, device=device)

with torch.no_grad():
    samples_check = model.generator(z_check).cpu()  # (64, 1, 28, 28)

# Per-pixel variance across the batch — high variance = diverse samples
per_pixel_var = samples_check.var(dim=0).mean().item()
print(f"Mean per-pixel variance across {n_check} samples: {per_pixel_var:.4f}")
print("(Values near 0 indicate mode collapse; healthy GANs typically show > 0.05)")

# Visual diversity check: plot all 64 samples on a grid
grid = samples_check[:64]
grid = (grid + 1) / 2

fig, axes = plt.subplots(8, 8, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(grid[i].squeeze(), cmap="gray")
    ax.axis("off")

plt.suptitle(f"64 GAN samples — per-pixel variance: {per_pixel_var:.4f}", fontsize=11)
plt.tight_layout()
plt.show()

---
## From DCGAN to StyleGAN2

The GAN we trained above is a **DCGAN** (Deep Convolutional GAN, Radford et al. 2015) — the architecture we used follows its design principles exactly: strided convolutions, BatchNorm, LeakyReLU in the discriminator, and ReLU + Tanh in the generator.

While DCGAN was a major step forward, it has two fundamental limitations:
- **Entangled latent space**: the noise vector $z$ controls all aspects of the image simultaneously — you cannot independently change style, pose, and fine detail.
- **Limited resolution**: training at high resolution is unstable without careful progressive techniques.

**StyleGAN2** (Karras et al. 2020) addresses both with three key innovations:

### 1. Mapping Network
Instead of feeding $z$ directly into the generator, a small MLP maps it to an intermediate latent space $\mathcal{W}$:
$$z \sim \mathcal{N}(0, I) \xrightarrow{f_{\text{map}}} w \in \mathcal{W}$$
The $\mathcal{W}$ space is empirically more *disentangled* than $\mathcal{Z}$ — directions in $\mathcal{W}$ tend to correspond to single semantic attributes (age, hair colour, pose).

### 2. Style Injection via AdaIN
The $w$ vector is injected into *every* layer of the synthesis network via **Adaptive Instance Normalisation (AdaIN)**:
$$\text{AdaIN}(x_i, w) = w_s \cdot \frac{x_i - \mu(x_i)}{\sigma(x_i)} + w_b$$
where $w_s, w_b$ are learned affine transforms of $w$. This lets different layers control different scales of detail (coarse: pose/shape, fine: texture/colour).

### 3. Weight Demodulation
StyleGAN2 replaces AdaIN with **weight demodulation** — scaling convolutional weights directly instead of normalising activations — which removes characteristic blob-like artefacts present in StyleGAN1.

The result: photorealistic face synthesis at 1024×1024 that was state-of-the-art in 2020 and remains a reference implementation.

### Setup

We use the **official NVlabs StyleGAN2-ADA PyTorch** implementation. The pretrained FFHQ model (~336 MB) is loaded directly from the NVIDIA CDN. No custom CUDA compilation is required for inference — the `.pkl` checkpoint bundles the full model definition via `torch_utils.persistence`.

The setup cell below clones the repo (to make `dnnlib` and `torch_utils` importable) and downloads the checkpoint. This takes about **1–2 minutes** on Colab.

In [ ]:
import subprocess, sys, os, io, warnings

# Clone StyleGAN2-ADA-PyTorch to get dnnlib + torch_utils
if not os.path.exists('stylegan2-ada-pytorch'):
    subprocess.run(['git', 'clone', '--depth=1',
                    'https://github.com/NVlabs/stylegan2-ada-pytorch.git'],
                   check=True)

# Make dnnlib and torch_utils importable
repo_path = os.path.abspath('stylegan2-ada-pytorch')
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

# Suppress Python warnings from the custom CUDA kernel fallback.
# On Python 3.12 / CUDA 12.x the custom ops (upfirdn2d, bias_act) cannot
# be JIT-compiled, so StyleGAN2 falls back to a pure-PyTorch reference
# implementation. This is slower but fully correct for inference.
warnings.filterwarnings('ignore', message='Failed to build CUDA kernels.*')

print('StyleGAN2-ADA repo ready.')

### Loading the Pretrained Model

We load the FFHQ 256×256 checkpoint pretrained by NVIDIA. The model is a full StyleGAN2-ADA generator trained on 70,000 high-quality human face images.

`G_ema` is the exponential moving average of the generator weights — it produces visually smoother results than the raw generator $G$ and is the standard choice for inference.

In [ ]:
import pickle, io
import torch
import numpy as np
import matplotlib.pyplot as plt
import urllib.request

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

PKL_URL  = 'https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl'
PKL_PATH = 'ffhq.pkl'

if not os.path.exists(PKL_PATH):
    print('Downloading FFHQ checkpoint (~336 MB)...')
    urllib.request.urlretrieve(PKL_URL, PKL_PATH)
    print('Done.')
else:
    print('Checkpoint already downloaded.')

with open(PKL_PATH, 'rb') as f:
    data = pickle.load(f)

G = data['G_ema'].to(device).eval()

print(f'Generator loaded. Output resolution: {G.img_resolution}x{G.img_resolution}')
print(f'Latent dim z: {G.z_dim},  Latent dim w: {G.w_dim}')

# Warmup: trigger CUDA plugin JIT compilation silently so that the
# "Setting up PyTorch plugin ... Failed!" prints don't appear later.
with torch.no_grad(), io.StringIO() as _buf:
    import sys as _sys
    _old_stdout, _sys.stdout = _sys.stdout, _buf
    try:
        _z = torch.zeros(1, G.z_dim, device=device)
        _l = torch.zeros(1, G.c_dim, device=device)
        G(_z, _l, truncation_psi=1.0, noise_mode='const')
    finally:
        _sys.stdout = _old_stdout

### Sampling Faces

We sample 16 noise vectors from $\mathcal{N}(0, I)$ and pass them through the full StyleGAN2 pipeline:
$$z \xrightarrow{\text{mapping network}} w \xrightarrow{\text{synthesis network (AdaIN)}} \text{image}$$

The **truncation trick** controls the trade-off between diversity and quality: $\psi < 1$ moves $w$ towards the mean $\bar{w}$, producing safer but less diverse faces. $\psi = 1$ samples from the full distribution.

In [ ]:
def stylegan2_sample(G, n: int, truncation_psi: float = 0.7, device: str = 'cpu'):
    """Sample n images from a StyleGAN2 generator."""
    z    = torch.randn(n, G.z_dim, device=device)
    label = torch.zeros(n, G.c_dim, device=device)  # unconditional
    with torch.no_grad():
        imgs = G(z, label, truncation_psi=truncation_psi, noise_mode='const')
    # Convert from [-1, 1] float to [0, 1] float
    imgs = (imgs.permute(0, 2, 3, 1).clamp(-1, 1) + 1) / 2
    return imgs.cpu().numpy()

samples = stylegan2_sample(G, n=16, truncation_psi=0.7, device=device)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(samples[i])
    ax.axis('off')

plt.suptitle('StyleGAN2 samples — FFHQ 1024×1024 (truncation $\\psi=0.7$)', y=1.02)
plt.tight_layout()
plt.show()

### Truncation Trick

The truncation trick interpolates the mapped $w$ towards the mean $\bar{w}$ of the $\mathcal{W}$ space:
$$w' = \bar{w} + \psi \cdot (w - \bar{w})$$

- $\psi = 1$: original sample — high diversity, occasional artefacts
- $\psi = 0.7$: standard choice — good quality and diversity balance
- $\psi = 0$: always generates the "average" face

Below we generate the same face with different $\psi$ values using a fixed seed.

In [ ]:
torch.manual_seed(42)
z_fixed = torch.randn(1, G.z_dim, device=device)
label   = torch.zeros(1, G.c_dim, device=device)

psi_values = [0.0, 0.3, 0.5, 0.7, 0.9, 1.0]

fig, axes = plt.subplots(1, len(psi_values), figsize=(14, 3))

with torch.no_grad():
    for ax, psi in zip(axes, psi_values):
        img = G(z_fixed, label, truncation_psi=psi, noise_mode='const')
        img = (img.squeeze().permute(1, 2, 0).clamp(-1, 1).cpu().numpy() + 1) / 2
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'$\\psi={psi}$', fontsize=10)

plt.suptitle('Effect of truncation $\\psi$ on the same latent code', y=1.02)
plt.tight_layout()
plt.show()

### Interpolation in $\mathcal{W}$ Space

The key advantage of StyleGAN2 over DCGAN for interpolation is that we interpolate in the **$\mathcal{W}$ space** rather than the raw $\mathcal{Z}$ space. Because the mapping network disentangles attributes, $\mathcal{W}$-space interpolations produce smoother and more semantically meaningful transitions.

In [ ]:
torch.manual_seed(0)
z_a = torch.randn(1, G.z_dim, device=device)
z_b = torch.randn(1, G.z_dim, device=device)
label = torch.zeros(1, G.c_dim, device=device)

n_steps = 8
alphas  = torch.linspace(0, 1, n_steps)

fig, axes = plt.subplots(1, n_steps, figsize=(16, 2.5))

with torch.no_grad():
    # Map both z vectors to W space
    w_a = G.mapping(z_a, label, truncation_psi=0.7)  # (1, num_ws, w_dim)
    w_b = G.mapping(z_b, label, truncation_psi=0.7)

    for ax, alpha in zip(axes, alphas):
        w_interp = (1 - alpha) * w_a + alpha * w_b
        img = G.synthesis(w_interp, noise_mode='const')
        img = (img.squeeze().permute(1, 2, 0).clamp(-1, 1).cpu().numpy() + 1) / 2
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f'{alpha:.2f}', fontsize=8)

plt.suptitle('Interpolation in $\\mathcal{W}$ space — StyleGAN2 FFHQ', y=1.05)
plt.tight_layout()
plt.show()

### Summary: DCGAN vs. StyleGAN2

| | DCGAN (notebook above) | StyleGAN2 |
|---|---|---|
| Latent input | $z$ fed directly to generator | $z \to w$ via mapping network |
| Style control | Entangled — $z$ controls everything at once | Disentangled $\mathcal{W}$ space per layer |
| Normalisation | BatchNorm | Weight demodulation |
| Resolution | 28×28 (Fashion-MNIST) | Up to 1024×1024 |
| Training | Minutes on T4 | Weeks on 8× V100 |
| Artefacts | Occasional mode collapse | Rare, no blob artefacts |

StyleGAN2 and its successor StyleGAN3 (which addresses aliasing) remain the standard reference architectures for high-fidelity image synthesis, and the $\mathcal{W}$-space disentanglement is the foundation for downstream applications such as GAN inversion, image editing, and face manipulation.